In [ ]:
import pandas as pd
import numpy as np
import re

#  File Load 
df_raw = pd.read_csv('data/raw/Parts.csv', sep=';', engine='python', on_bad_lines='warn')

# Columns list 
correct_columns = [
    'ID', 'DESCRIPTION', 'Attribut1', 'Additional Feature', 'Application', 
    'Characteristic', 'Temp', 'Height', 'Length in mm', 'Rating', 'Material', 
    'Size', 'Code', 'Joule-integral-Nom (J)', 'LC Risk', 'Maximum AC Voltage Rating', 
    'Maximum DC Voltage Rating', 'Maximum Power Dissipation', 'Mounting', 
    'Mounting Feature', 'Number of Terminals', 'Operating Temperature-Max (Cel)', 
    'Operating Temperature-Min (Cel)', 'Physical Dimension', 'Pre-arcing time-Min (ms)', 
    'Product Diameter', 'Product Length', 'Rated Breaking Capacity (A)', 
    'Rated Current (A)', 'Rated Voltage (V)', 'Rated Voltage(AC) (V)', 'Rated Voltage(DC) (V)'
]

# Check columns mismatched 
if len(df_raw.columns) == len(correct_columns):
    df_raw.columns = correct_columns

In [ ]:
def smart_aligner(row):
    # Temperature Column Fix
    if pd.notna(row['Temp']) and 'mm' in str(row['Temp']):
        val = row['Temp']
        # Height shift from temp column
        if pd.isna(row['Height']) or row['Height'] == "":
            row['Height'] = val
            row['Temp'] = np.nan

    # Rated Voltage Shift
    # Extract voltage from Description
    if pd.isna(row['Rated Voltage (V)']):
        desc = str(row['DESCRIPTION'])
        match = re.search(r'(\d+)V', desc)
        if match:
            row['Rated Voltage (V)'] = match.group(1)

    # Rated Current Shift
    # Extract Current from Description
    if pd.isna(row['Rated Current (A)']):
        desc = str(row['DESCRIPTION'])
        match = re.search(r'(\d+\.?\d*)A', desc)
        if match:
            row['Rated Current (A)'] = match.group(1)

    return row

# Function
df_cleaned = df_raw.apply(smart_aligner, axis=1)

In [ ]:
# Check Height column
print("--- Height Column Post-Fix ---")
print(df_cleaned['Height'].dropna().head())

# Check Temp column
print("\n--- Temp Column Post-Fix ---")
print(df_cleaned['Temp'].unique())

--- Height Column Post-Fix ---
0    20mm
1    20mm
2    20mm
3    20mm
4    20mm
Name: Height, dtype: object

--- Temp Column Post-Fix ---
[nan '5.03mm' '6.35mm' '5.1mm' '5.08mm' '0.61mm' '1.52mm' '0.81mm'
 '0.813mm' '0.51mm' '0.56mm' '1.63mm' '0.85mm' '0.83mm' '0.88mm' '3.12mm'
 '2.69mm']


In [7]:
import pandas as pd
import numpy as np
import re

# --- HELPER FUNCTION ---
# Yeh function text se numbers nikalne ke liye hai
def extract_number(val):
    if pd.isna(val): return np.nan
    res = re.findall(r"[-+]?\d*\.\d+|\d+", str(val))
    return float(res[0]) if res else np.nan

# --- THE DOMAIN DISPATCHER ---
def domain_dispatcher(row):
    # Har row ki saari values ko scan karte hain (Semicolon separation ke baad)
    all_values = [str(val) for val in row.values if pd.notna(val)]
    
    for item in all_values:
        item_clean = item.strip()
        
        # 1. TEMPERATURE LOGIC (Cel)
        if 'Cel' in item_clean or '°C' in item_clean:
            num = extract_number(item_clean)
            # Industrial Standard: 125Cel Max hota hai aur -55Cel Min
            if num > 40:
                row['Operating Temperature-Max (Cel)'] = num
            else:
                row['Operating Temperature-Min (Cel)'] = num
        
        # 2. DIMENSION LOGIC (mm)
        elif 'mm' in item_clean:
            num = extract_number(item_clean)
            # Domain Rules: 
            # 15mm+ is usually Length
            # 4mm to 14.9mm is usually Product Diameter
            # <4mm is usually Height or Lead Thickness
            if num >= 15:
                row['Product Length'] = num
            elif 4 <= num < 15:
                row['Product Diameter'] = num
            else:
                row['Height'] = num
        
        # 3. CURRENT LOGIC (A)
        # Regex to match '1.6A', '8A', but NOT 'VAC' or 'VDC'
        elif re.search(r'^\d+\.?\d*A$', item_clean):
            row['Rated Current (A)'] = extract_number(item_clean)
            
        # 4. VOLTAGE LOGIC (V)
        # Matches '250V', '125V'
        elif re.search(r'^\d+V$', item_clean):
            row['Rated Voltage (V)'] = extract_number(item_clean)

    return row

# --- EXECUTION ---
# 1. Apply the alignment logic
df_final = df_raw.apply(domain_dispatcher, axis=1)

# 2. Cleanup: Misaligned 'Temp' column drop karein (Data ab Height/Diameter mein hai)
if 'Temp' in df_final.columns:
    df_final = df_final.drop(columns=['Temp'])

print("✅ Dispatcher run successful. No NameErrors this time!")
print(df_final[['Product Diameter', 'Product Length', 'Height']].head())

✅ Dispatcher run successful. No NameErrors this time!
   Product Diameter Product Length Height
0               5.2           20.0    1.6
1               5.2           20.0   20mm
2               5.2           20.0   20mm
3               5.2           20.0   20mm
4               5.2           20.0   20mm


In [9]:
import pandas as pd
import numpy as np
import re

# --- STEP 1: LOAD & HEADER FIX ---
# Semicolon (;) separator use karke load karenge aur extra commas headers se saaf karenge
df_raw = pd.read_csv('data/raw/Parts.csv', sep=';', engine='python')
df_raw.columns = [re.sub(r'[,]+', '', col).strip() for col in df_raw.columns]

# --- STEP 2: HELPER FUNCTIONS ---
def get_numeric(val):
    """Text se float nikalne ke liye (e.g., '1.6A' -> 1.6)"""
    if pd.isna(val) or val == "": return np.nan
    res = re.findall(r"[-+]?\d*\.\d+|\d+", str(val))
    return float(res[0]) if res else np.nan

# --- STEP 3: LOGIC-BASED DISPATCHER ---
def clean_and_align(row):
    # Har row ki saari values ko ek scan list mein daalte hain
    all_vals = [str(x).strip() for x in row.values if pd.notna(x)]
    
    # 1. Dimensions (mm)
    # Logic: 15mm+ is Length, 4-10mm is Diameter, <4mm is Height/Lead
    mm_vals = [v for v in all_vals if 'mm' in v.lower()]
    for v in mm_vals:
        n = get_numeric(v)
        if n >= 15: row['Product Length'] = n
        elif 4 <= n < 15: row['Product Diameter'] = n
        else: row['Height'] = n

    # 2. Temperature (Cel)
    cel_vals = [v for v in all_vals if 'cel' in v.lower() or '°c' in v.lower()]
    for v in cel_vals:
        n = get_numeric(v)
        if n > 40: row['Operating Temperature-Max (Cel)'] = n
        else: row['Operating Temperature-Min (Cel)'] = n

    # 3. Electrical (A & V)
    for v in all_vals:
        if re.search(r'^\d+\.?\d*A$', v): # Rated Current
            row['Rated Current (A)'] = get_numeric(v)
        if re.search(r'^\d+V$', v) and 'VAC' not in v: # Rated Voltage
            row['Rated Voltage (V)'] = get_numeric(v)
            
    return row

# Alignment Apply karte hain
df_cleaned = df_raw.apply(clean_and_align, axis=1)

# --- STEP 4: DESCRIPTION MINING (RECOVERING NULLS) ---
def recover_from_description(row):
    desc = str(row['DESCRIPTION']).lower()
    
    # Recover Current if NaN (e.g., '1.6A' in desc)
    if pd.isna(row['Rated Current (A)']):
        match = re.search(r'(\d+\.?\d*)a\b', desc)
        if match: row['Rated Current (A)'] = float(match.group(1))
        
    # Recover Dimensions (e.g., '5x20mm' in desc)
    if pd.isna(row['Product Length']) or pd.isna(row['Product Diameter']):
        match = re.search(r'(\d+\.?\d*)\s*[x*]\s*(\d+\.?\d*)mm', desc)
        if match:
            row['Product Diameter'] = float(match.group(1))
            row['Product Length'] = float(match.group(2))
            
    return row

df_cleaned = df_cleaned.apply(recover_from_description, axis=1)

# --- STEP 5: FINAL POLISH ---
# Unnecessary/Shifted columns drop karna
cols_to_drop = ['Temp', 'Rating'] 
df_cleaned = df_cleaned.drop(columns=[c for c in cols_to_drop if c in df_cleaned.columns])

# Units ko numerical values mein convert karna pure dataset ke liye
numeric_cols = [
    'Product Diameter', 'Product Length', 'Height', 'Rated Current (A)', 
    'Rated Voltage (V)', 'Operating Temperature-Max (Cel)', 
    'Operating Temperature-Min (Cel)', 'Joule-integral-Nom (J)'
]

for col in numeric_cols:
    if col in df_cleaned.columns:
        df_cleaned[col] = df_cleaned[col].apply(get_numeric)

print("✅ Data Cleaning & Alignment Successful!")
print(df_cleaned[['Product Diameter', 'Product Length', 'Rated Current (A)', 'Rated Voltage (V)']].head())

✅ Data Cleaning & Alignment Successful!
   Product Diameter  Product Length  Rated Current (A)  Rated Voltage (V)
0               5.2            20.0                1.6              250.0
1               5.2            20.0                6.3              250.0
2               5.2            20.0                8.0              250.0
3               5.2            20.0               10.0              250.0
4               5.2            20.0               12.5              250.0


In [ ]:
import pandas as pd
import numpy as np
import re

# --- extract numbers  ---
def extract_pure_number(val):
    if pd.isna(val) or val == "": return np.nan
    # Text se number nikalna (e.g., '1500A' -> 1500, '0.755J' -> 0.755)
    res = re.findall(r"[-+]?\d*\.\d+|\d+", str(val))
    return float(res[0]) if res else np.nan

# ---  MULTI-COLUMN DISPATCHER ---
def final_structure_fix(row):
    all_vals = [str(x).strip() for x in row.values if pd.notna(x)]
    
    for v in all_vals:
        # A. Joule-Integral Recovery (J)
        if 'j' in v.lower() and not any(unit in v.lower() for unit in ['mm', 'cel', 'a', 'v']):
            row['Joule-integral-Nom (J)'] = extract_pure_number(v)
            
        # B. Breaking Capacity (A) 
        if 'a' in v.lower() and 'ir' in v.lower():
            row['Rated Breaking Capacity (A)'] = extract_pure_number(v)
            
        # C. Voltage Fix (AC vs DC)
        if 'vac' in v.lower():
            row['Rated Voltage(AC) (V)'] = extract_pure_number(v)
        if 'vdc' in v.lower():
            row['Rated Voltage(DC) (V)'] = extract_pure_number(v)
            
    return row

df_structured = df_cleaned.apply(final_structure_fix, axis=1)

# --- DATA TYPE CONVERSION ---
numeric_targets = [
    'Product Diameter', 'Product Length', 'Height', 
    'Rated Current (A)', 'Rated Voltage (V)', 
    'Rated Voltage(AC) (V)', 'Rated Voltage(DC) (V)',
    'Rated Breaking Capacity (A)', 'Joule-integral-Nom (J)',
    'Operating Temperature-Max (Cel)', 'Operating Temperature-Min (Cel)'
]

for col in numeric_targets:
    if col in df_structured.columns:
        df_structured[col] = df_structured[col].apply(extract_pure_number)
        df_structured[col] = pd.to_numeric(df_structured[col], errors='coerce')

# String/Object 
categorical_targets = ['DESCRIPTION', 'Attribut1', 'Application', 'Characteristic', 'Material', 'Mounting']
for col in categorical_targets:
    if col in df_structured.columns:
        df_structured[col] = df_structured[col].astype(str).replace('nan', np.nan)

print("✅ Data Structure & Values Fixed!")
print("\n--- Current Data Types ---")
print(df_structured[numeric_targets].dtypes)

✅ Data Structure & Values Fixed!

--- Current Data Types ---
Product Diameter                   float64
Product Length                     float64
Height                             float64
Rated Current (A)                  float64
Rated Voltage (V)                  float64
Rated Voltage(AC) (V)              float64
Rated Voltage(DC) (V)              float64
Rated Breaking Capacity (A)        float64
Joule-integral-Nom (J)             float64
Operating Temperature-Max (Cel)    float64
Operating Temperature-Min (Cel)    float64
dtype: object


In [ ]:
# checking columns - validations
cat_cols = ['Characteristic', 'Material', 'Mounting', 'Attribut1', 'Application']

print("--- Unique Values Inspection ---")
for col in cat_cols:
    if col in df_structured.columns:
        print(f"\nColumn: {col}")
        # Check unique values
        print(df_structured[col].astype(str).str.upper().str.strip().unique()[:15])

--- Unique Values Inspection ---

Column: Characteristic
['VERY FAST' 'NAN' 'TIME LAG' 'SUPER FAST' 'FAST' 'MEDIUM TIME LAG' 'SLOW'
 'MEDIUM']

Column: Material
['CERAMIC' 'GLASS' 'NAN' 'THERMOPLASTIC' 'EPOXY COATED' 'POLYAMIDE 66'
 'NYLON' 'EPOXY' 'EPOXY GLASS']

Column: Mounting
['HOLDER' 'NAN' 'THROUGH HOLE' 'PANEL MOUNT' 'SURFACE MOUNT' 'SOCKET']

Column: Attribut1
['FAST' 'NAN' 'SLOW BLOW' 'VERY FAST' 'MEDIUM TIME DELAY' 'SUPER TIME LAG']

Column: Application
['PRIMARY PROTECTION IN EQUIPMENT' 'INDUSTRIAL ELECTRONIC'
 'POWER SUPPLY ADAPTER|PRIMARY PROTECTION ON PCB|SWITCHING MODE POWER SUPPLY'
 'POWER SUPPLY ADAPTER|PRIMARY PROTECTION IN EQUIPMENT|SWITCHING MODE POWER SUPPLY'
 'PRIMARY PROTECTION ON PCB|POWER SUPPLY ADAPTER FOR LAPTOP|SWITCHING MODE POWER SUPPLY'
 'NAN' 'MOTOR CIRCUIT' 'SEMICONDUCTOR PROTECTION|TEST EQUIPMENT'
 'PRIMARY PROTECTION ON PCB' 'PRIMARY PROTECTION ON SMD PCB'
 'POWER SUPPLY ADAPTER FOR LAPTOP|PRIMARY PROTECTION ON PCB|SWITCHING MODE POWER SUPPLY'
 'SUPP

In [ ]:
# --- 1. CATEGORICAL MAPPING (Standardization) ---
char_mapping = {
    'VERY FAST': 'VERY FAST',
    'SUPER FAST': 'VERY FAST',
    'TIME LAG': 'SLOW',
    'SLOW BLOW': 'SLOW',
    'SUPER TIME LAG': 'SLOW',
    'MEDIUM TIME LAG': 'MEDIUM',
    'MEDIUM TIME DELAY': 'MEDIUM'
}

# Apply Mapping
df_structured['Characteristic'] = df_structured['Characteristic'].str.upper().replace(char_mapping)
df_structured['Attribut1'] = df_structured['Attribut1'].str.upper().replace(char_mapping)

# --- 2. APPLICATION CLEANING (Handling Multi-values) ---
def clean_application(val):
    if pd.isna(val) or val == 'NAN': return np.nan
    parts = str(val).split('|')
    return parts[0].strip().upper()

df_structured['Application'] = df_structured['Application'].apply(clean_application)

# --- 3. MATERIAL & MOUNTING  ---
for col in ['Material', 'Mounting']:
    df_structured[col] = df_structured[col].str.upper().str.strip()
    if col == 'Material':
        df_structured[col] = df_structured[col].replace({'EPOXY GLASS': 'EPOXY', 'EPOXY COATED': 'EPOXY'})

# --- 4. NON-CATEGORICAL (Description & ID) ---
df_structured['DESCRIPTION'] = df_structured['DESCRIPTION'].astype(str).str.strip()
df_structured['ID'] = df_structured['ID'].astype(str)

print("✅ Categorical & Non-Categorical Data is now Standardized!")
print(df_structured[['Characteristic', 'Material', 'Application']].head())

✅ Categorical & Non-Categorical Data is now Standardized!
  Characteristic Material                      Application
0      VERY FAST  CERAMIC  PRIMARY PROTECTION IN EQUIPMENT
1      VERY FAST  CERAMIC  PRIMARY PROTECTION IN EQUIPMENT
2      VERY FAST  CERAMIC  PRIMARY PROTECTION IN EQUIPMENT
3      VERY FAST  CERAMIC  PRIMARY PROTECTION IN EQUIPMENT
4      VERY FAST  CERAMIC  PRIMARY PROTECTION IN EQUIPMENT


In [ ]:
# 1. Check Data Types of all columns
print("--- 1. Data Types Audit ---")
print(df_structured.dtypes)

# 2. Check for 'Non-Numeric' entries in Numeric Columns 
numeric_cols = [
    'Product Diameter', 'Product Length', 'Rated Current (A)', 
    'Rated Voltage (V)', 'Operating Temperature-Max (Cel)'
]

print("\n--- 2. Numeric Column Health ---")
for col in numeric_cols:
    if df_structured[col].dtype == 'object':
        print(f"⚠️ ALERT: {col} is still an OBJECT. Checking for dirty values...")
        # Dirty values check karein (jo numbers mein convert nahi ho paaye)
        dirty = df_structured[col][pd.to_numeric(df_structured[col], errors='coerce').isna()].unique()
        print(f"   Dirty Values found: {dirty[:5]}")
    else:
        print(f"✅ {col} is correctly stored as {df_structured[col].dtype}")

# 3. Check for 'Numeric' leakage in Categorical Columns

cat_cols = ['Characteristic', 'Material', 'Mounting']
print("\n--- 3. Categorical Column Health ---")
for col in cat_cols:
    # Check if any value in categorical column contains a digit/number
    leaked_numeric = df_structured[col][df_structured[col].astype(str).str.contains(r'\d', na=False)].unique()
    if len(leaked_numeric) > 0:
        print(f"⚠️ ALERT: {col} has numeric-looking values: {leaked_numeric[:5]}")
    else:
        print(f"✅ {col} is clean (No numeric leakage).")
        

--- 1. Data Types Audit ---
ID                                  object
DESCRIPTION                         object
Attribut1                           object
Additional Feature                  object
Application                         object
Characteristic                      object
Height                             float64
Length in mm                        object
Material                            object
Size                                object
Code                                object
Joule-integral-Nom (J)             float64
LC Risk                             object
Maximum AC Voltage Rating           object
Maximum DC Voltage Rating           object
Maximum Power Dissipation           object
Mounting                            object
Mounting Feature                    object
Number of Terminals                float64
Operating Temperature-Max (Cel)    float64
Operating Temperature-Min (Cel)    float64
Physical Dimension                  object
Pre-arcing time-Min (ms)  

In [ ]:
# 1. List of remaining technical columns that should be numeric
remaining_numeric = [
    'Length in mm', 
    'Maximum AC Voltage Rating', 
    'Maximum DC Voltage Rating', 
    'Maximum Power Dissipation', 
    'Pre-arcing time-Min (ms)'
]

# 2. Convert to numeric 
for col in remaining_numeric:
    if col in df_structured.columns:
        
        df_structured[col] = df_structured[col].apply(extract_pure_number)
        df_structured[col] = pd.to_numeric(df_structured[col], errors='coerce')

# 3. Final Type Check to confirm everything is float64 now
print("--- Final Technical Columns Check ---")
print(df_structured[remaining_numeric].dtypes)


--- Final Technical Columns Check ---
Length in mm                 float64
Maximum AC Voltage Rating    float64
Maximum DC Voltage Rating    float64
Maximum Power Dissipation    float64
Pre-arcing time-Min (ms)     float64
dtype: object


In [15]:
# 1. Dataset ka structure aur samples dekhna
print("--- 📂 Dataset Preview (First 5 Rows) ---")
display(df_structured.head())

print("\n--- 📂 Dataset Preview (Last 5 Rows) ---")
display(df_structured.tail())

print("\n--- 🎲 Random Sample (10 Rows) ---")
display(df_structured.sample(10))


--- 📂 Dataset Preview (First 5 Rows) ---


,ID,DESCRIPTION,Attribut1,Additional Feature,Application,Characteristic,Height,Length in mm,Material,Size,...,Operating Temperature-Min (Cel),Physical Dimension,Pre-arcing time-Min (ms),Product Diameter,Product Length,Rated Breaking Capacity (A),Rated Current (A),Rated Voltage (V),Rated Voltage(AC) (V),Rated Voltage(DC) (V)
0,A1,Indicator Red Fast Movement 1.6A 250V Holder P...,FAST,NaN,PRIMARY PROTECTION IN EQUIPMENT,VERY FAST,1.6,5.2,CERAMIC,5 X 20mm,...,55.0,5.2mm x 20mm,3.0,5.2,20.0,1.6,1.6,250.0,1.6,NaN
1,A2,"Non Resettable Indicators Electric Indicator, ...",NaN,NaN,PRIMARY PROTECTION IN EQUIPMENT,VERY FAST,20.0,5.2,CERAMIC,5 X 20mm,...,55.0,5.2mm x 20mm,3.0,5.2,20.0,6.3,6.3,250.0,6.3,NaN
2,A3,Indicator Red Fast Movement 8A 250V Holder Pla...,FAST,NaN,PRIMARY PROTECTION IN EQUIPMENT,VERY FAST,20.0,5.2,CERAMIC,5 X 20mm,...,55.0,5.2mm x 20mm,10.0,5.2,20.0,8.0,8.0,250.0,8.0,NaN
3,A4,"Non Resettable Indicators Electric Indicator, ...",NaN,NaN,PRIMARY PROTECTION IN EQUIPMENT,VERY FAST,20.0,5.2,CERAMIC,5 X 20mm,...,55.0,5.2mm x 20mm,10.0,5.2,20.0,10.0,10.0,250.0,10.0,NaN
4,A5,Indicator Red Fast Movement 12.5A 250V Holder ...,FAST,NaN,PRIMARY PROTECTION IN EQUIPMENT,VERY FAST,20.0,5.2,CERAMIC,5 X 20mm,...,55.0,5.2mm x 20mm,10.0,5.2,20.0,12.5,12.5,250.0,12.5,NaN



--- 📂 Dataset Preview (Last 5 Rows) ---


,ID,DESCRIPTION,Attribut1,Additional Feature,Application,Characteristic,Height,Length in mm,Material,Size,...,Operating Temperature-Min (Cel),Physical Dimension,Pre-arcing time-Min (ms),Product Diameter,Product Length,Rated Breaking Capacity (A),Rated Current (A),Rated Voltage (V),Rated Voltage(AC) (V),Rated Voltage(DC) (V)
993,A994,Indicator Chip Slow Blow Movement 1.5A 125V SM...,SLOW,NaN,AUTOMOTIVE,NaN,NaN,NaN,CERAMIC,6.1 X 2.69mm,...,NaN,NaN,NaN,6.1,6.1,NaN,1.5,125.0,NaN,NaN
994,A995,Indicator Chip Slow Blow Movement 1.5A 125V SM...,SLOW,RATED BREAKING CAPACITY AT 125 VDC: 50 A,AUTOMOTIVE,SLOW,2.69,6.1,CERAMIC,6.1 X 2.69mm,...,55.0,6.1mm x 2.69mm x 2.69mm,3.65,6.1,6.1,NaN,1.5,125.0,1.5,125.0
995,A996,Indicator Chip Slow Blow Movement 2.5A 125V SM...,SLOW,RATED BREAKING CAPACITY AT 125 VDC: 50 A,AUTOMOTIVE,SLOW,2.69,6.1,CERAMIC,6.1 X 2.69mm,...,55.0,6.1mm x 2.69mm x 2.69mm,NaN,6.1,6.1,NaN,2.5,125.0,2.5,125.0
996,A997,Indicator Chip Slow Blow Movement 2.5A 125V SM...,SLOW,RATED BREAKING CAPACITY AT 125 VDC: 50 A,AUTOMOTIVE,SLOW,2.69,6.1,CERAMIC,6.1 X 2.69mm,...,55.0,6.1mm x 2.69mm x 2.69mm,15.00,6.1,6.1,NaN,2.5,125.0,2.5,125.0
997,A998,Indicator Chip Slow Blow Movement 2.5A 125V SM...,SLOW,RATED BREAKING CAPACITY AT 125 VDC: 50 A,AUTOMOTIVE,SLOW,2.69,6.1,CERAMIC,6.1 X 2.69mm,...,55.0,6.1mm x 2.69mm x 2.69mm,15.00,6.1,6.1,NaN,2.5,125.0,2.5,125.0



--- 🎲 Random Sample (10 Rows) ---


,ID,DESCRIPTION,Attribut1,Additional Feature,Application,Characteristic,Height,Length in mm,Material,Size,...,Operating Temperature-Min (Cel),Physical Dimension,Pre-arcing time-Min (ms),Product Diameter,Product Length,Rated Breaking Capacity (A),Rated Current (A),Rated Voltage (V),Rated Voltage(AC) (V),Rated Voltage(DC) (V)
351,A352,nan,FAST,NaN,MOTOR CIRCUIT,NaN,NaN,NaN,GLASS,6.3 X 32mm,...,NaN,NaN,NaN,14.48,14.48,NaN,8.00,250.0,NaN,NaN
537,A538,Indicator Chip Fast Movement 3A SMD Solder Pad...,FAST,NaN,MOTOR CIRCUIT,FAST,1.00,NaN,NaN,1 X 0.51mm,...,NaN,NaN,NaN,NaN,1.00,3.0,3.00,NaN,NaN,3.0
586,A587,nan,VERY FAST,NaN,BATTERY PACK,NaN,3.18,NaN,EPOXY,3.18 X 1.52 X 0.58mm,...,NaN,NaN,NaN,NaN,3.18,NaN,2.00,63.0,NaN,NaN
19,A20,Indicator Red Slow Blow Movement 16A 250V Hold...,SLOW,RATED BREAKING CAPACITY AT 125 VDC: 1500 A,POWER SUPPLY ADAPTER,SLOW,20.00,5.2,CERAMIC,5x20mm,...,55.0,5.2mm x 20mm,20.0,5.20,20.00,16.0,16.00,250.0,16.0,125.0
32,A33,"Indicators PN Electric Indicator, Time Lag Blo...",NaN,RATED BREAKING CAPACITY AT 300 VDC: 1500 A,NaN,SLOW,20.00,5.2,NaN,5 X 20mm,...,55.0,5.2mm x 20mm,10.0,5.20,20.00,2.0,2.00,250.0,2.0,300.0
138,A139,nan,SLOW,NaN,MOTOR CIRCUIT,NaN,NaN,NaN,CERAMIC,6.3 X 32mm,...,NaN,NaN,NaN,6.30,NaN,NaN,10.00,250.0,NaN,NaN
143,A144,nan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
265,A266,Indicator Red Fast Movement 0.1A 250V Holder P...,FAST,NaN,MOTOR CIRCUIT,FAST,0.10,5.2,CERAMIC,5 X 20mm,...,55.0,5.2mm x 20mm,NaN,5.20,20.00,0.1,0.10,250.0,0.1,NaN
442,A443,nan,SLOW,NaN,AMPLIFIER,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,5.20,20.00,NaN,5.00,32.0,NaN,NaN
572,A573,Indicator Chip Slow Blow Movement 3.15A 32V SM...,SLOW,NaN,MOTOR CIRCUIT,NaN,3.18,NaN,EPOXY,3.18 X 1.52 X 1.14mm,...,NaN,NaN,NaN,NaN,3.18,NaN,3.15,32.0,NaN,NaN


In [16]:
print(f"Total Columns: {len(df_structured.columns)}")
print("-" * 30)
print(list(df_structured.columns))

Total Columns: 30
------------------------------
['ID', 'DESCRIPTION', 'Attribut1', 'Additional Feature', 'Application', 'Characteristic', 'Height', 'Length in mm', 'Material', 'Size', 'Code', 'Joule-integral-Nom (J)', 'LC Risk', 'Maximum AC Voltage Rating', 'Maximum DC Voltage Rating', 'Maximum Power Dissipation', 'Mounting', 'Mounting Feature', 'Number of Terminals', 'Operating Temperature-Max (Cel)', 'Operating Temperature-Min (Cel)', 'Physical Dimension', 'Pre-arcing time-Min (ms)', 'Product Diameter', 'Product Length', 'Rated Breaking Capacity (A)', 'Rated Current (A)', 'Rated Voltage (V)', 'Rated Voltage(AC) (V)', 'Rated Voltage(DC) (V)']


In [ ]:
# Unique values report
print("--- 🔍 Unique Values Audit Report ---\n")

for col in df_structured.columns:
    unique_vals = df_structured[col].dropna().unique()
    num_unique = len(unique_vals)
    
    print(f"Column: 【 {col} 】")
    print(f"Unique Values Count: {num_unique}")
    
    # categorical unique values
    if num_unique > 20:
        print(f"Sample Values: {list(unique_vals[:10])} ... [and {num_unique-10} more]")
    else:
        print(f"All Unique Values: {list(unique_vals)}")
    
    print("-" * 50)

--- 🔍 Unique Values Audit Report ---

Column: 【 ID 】
Unique Values Count: 998
Sample Values: ['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9', 'A10'] ... [and 988 more]
--------------------------------------------------
Column: 【 DESCRIPTION 】
Unique Values Count: 583
Sample Values: ['Indicator Red Fast Movement 1.6A 250V Holder Plastic 5 X 20mm Ceramic Box CCC/PSE/VDE/cULus Electric Indicator, Very Fast Blow, 1.6A, 250VAC, 1500A (IR), Inline/holder, 5x20mm', 'Non Resettable Indicators Electric Indicator, Very Fast Blow, 6.3A, 250VAC, 1500A (IR), Inline/holder, 5x20mm', 'Indicator Red Fast Movement 8A 250V Holder Plastic 5 X 20mm Ceramic Box KC/PSE/VDE/cULus Electric Indicator, Very Fast Blow, 8A, 250VAC, 1500A (IR), Inline/holder, 5x20mm', 'Non Resettable Indicators Electric Indicator, Very Fast Blow, 10A, 250VAC, 1500A (IR), Inline/holder, 5x20mm', 'Indicator Red Fast Movement 12.5A 250V Holder Plastic 5 X 20mm Ceramic Box PSE/cULus Electric Indicator, Very Fast Blow, 12.5A, 250

In [ ]:
def surgical_fix(row):
    # 1. Rated Voltage (AC) 
    v_ac = row['Rated Voltage(AC) (V)']
    if pd.notna(v_ac) and v_ac < 50:
        if pd.isna(row['Rated Current (A)']):
            row['Rated Current (A)'] = v_ac
        row['Rated Voltage(AC) (V)'] = np.nan # Galat jagah se delete karein

    # 2. Height data redistribute 
    h_val = row['Height']
    if pd.notna(h_val):
        if h_val > 15: 
            row['Product Length'] = h_val
        elif 0.1 <= h_val <= 15: 
    
            if pd.isna(row['Rated Current (A)']):
                row['Rated Current (A)'] = h_val
        row['Height'] = np.nan # Clean the messy column

    # 3. Breaking Capacity Fix
    if row['Rated Breaking Capacity (A)'] == row['Rated Current (A)']:
        row['Rated Breaking Capacity (A)'] = np.nan

    return row

# Apply the surgical fix
df_structured = df_structured.apply(surgical_fix, axis=1)

# Cleanup redundant columns
cols_to_drop = ['Height', 'Length in mm', 'Physical Dimension']
df_structured = df_structured.drop(columns=[c for c in cols_to_drop if c in df_structured.columns])

print("✅ Surgical Alignment Complete!")
print("--- Final Cleaned Column Types ---")
print(df_structured[['Rated Current (A)', 'Rated Voltage(AC) (V)', 'Product Length']].describe())

✅ Surgical Alignment Complete!
--- Final Cleaned Column Types ---
       Rated Current (A)  Rated Voltage(AC) (V)  Product Length
count         945.000000              64.000000      854.000000
mean            3.631602             179.796875       12.290574
std             4.624885              70.685797        9.558208
min             0.002000              63.000000        0.500000
25%             0.750000             125.000000        6.100000
50%             2.000000             125.000000        8.400000
75%             5.000000             250.000000       20.000000
max            35.000000             250.000000       49.000000


In [ ]:
# 1. Total Rows Check
print(f"--- 📊 Volume Integrity ---")
print(f"Raw Data Rows: {len(df_raw)}")
print(f"Cleaned Data Rows: {len(df_final)}")
print(f"Match: {'✅' if len(df_raw) == len(df_final) else '❌'}")

# 2. Information Recovery Check (Current)
raw_current_nulls = df_raw['Rating'].isnull().sum() # Assuming 'Rating' was the original column
clean_current_filled = df_final['Rated Current (A)'].notnull().sum()

print(f"\n--- ⚡ Electrical Specs Integrity ---")
print(f"Rows with valid Current in Cleaned Data: {clean_current_filled}")

# 3. Consistency Sample Check (Side-by-Side)
# checking random with description columns
print(f"\n--- 🔍 Spot-Check (Raw Description vs Cleaned Specs) ---")
sample_check = df_final[['DESCRIPTION', 'Rated Current (A)', 'Rated Voltage (V)', 'Product Length']].sample(5)
print(sample_check)

# 4. Outlier & Range Check (Sanity)
print(f"\n--- 🌡️ Logical Consistency ---")
# Check if any Current is impossibly high or Voltage is too low
logical_errors = df_final[(df_final['Rated Current (A)'] > 1000) | (df_final['Rated Voltage (V)'] < 1)].shape[0]
print(f"Logical Errors found: {logical_errors} {'✅ (Clean)' if logical_errors == 0 else '⚠️ (Check Data)'}")

--- 📊 Volume Integrity ---
Raw Data Rows: 998
Cleaned Data Rows: 998
Match: ✅

--- ⚡ Electrical Specs Integrity ---
Rows with valid Current in Cleaned Data: 930

--- 🔍 Spot-Check (Raw Description vs Cleaned Specs) ---
                                           DESCRIPTION  Rated Current (A)  \
463  Indicator Red Fast Movement 0.031A 250V Holder...              0.031   
913  Indicator Chip Very Fast Movement 0.1A 125V SM...              0.100   
989  Indicator Chip Slow Blow Movement 7A 72V SMD S...              7.000   
771  Indicator Chip 3.5A 32V SMD Solder Pad 0603 Ce...              3.500   
28   Indicator Red Slow Blow Movement 1.25A 250V Ho...              1.250   

     Rated Voltage (V) Product Length  
463              250.0          31.75  
913              125.0          6.1mm  
989               72.0          6.1mm  
771               32.0         1.54mm  
28               250.0           20.0  

--- 🌡️ Logical Consistency ---
Logical Errors found: 0 ✅ (Clean)


In [ ]:
# 1. Temperature Stats Check
print("--- 🌡️ Temperature Column Stats ---")
temp_cols = ['Operating Temperature-Min (Cel)', 'Operating Temperature-Max (Cel)']
print(df_structured[temp_cols].describe())

# 2. Logical Consistency Check (Max should be > Min)
temp_errors = df_structured[df_structured['Operating Temperature-Max (Cel)'] <= df_structured['Operating Temperature-Min (Cel)']]

if not temp_errors.empty:
    print(f"\n⚠️ ALERT: Found {len(temp_errors)} rows where Max Temp is not greater than Min Temp!")
    print(temp_errors[['ID', 'DESCRIPTION'] + temp_cols].head())
else:
    print("\n✅ Logic Pass: All Max Temperatures are greater than Min Temperatures.")

# 3. Frequency Check (Unique Values)
print("\n--- 📊 Unique Temperature Points ---")
print("Min Temp Values:", df_structured['Operating Temperature-Min (Cel)'].unique())
print("Max Temp Values:", df_structured['Operating Temperature-Max (Cel)'].unique())

--- 🌡️ Temperature Column Stats ---
       Operating Temperature-Min (Cel)  Operating Temperature-Max (Cel)
count                       636.000000                       707.000000
mean                         54.410377                        56.923621
std                           2.917202                         9.784062
min                          40.000000                        55.000000
25%                          55.000000                        55.000000
50%                          55.000000                        55.000000
75%                          55.000000                        55.000000
max                          55.000000                       125.000000

⚠️ ALERT: Found 611 rows where Max Temp is not greater than Min Temp!
   ID                                        DESCRIPTION  \
0  A1  Indicator Red Fast Movement 1.6A 250V Holder P...   
1  A2  Non Resettable Indicators Electric Indicator, ...   
2  A3  Indicator Red Fast Movement 8A 250V Holder Pla...   
3  A4

In [ ]:
def fix_temperature_logic(row):
    min_t = row['Operating Temperature-Min (Cel)']
    max_t = row['Operating Temperature-Max (Cel)']
    desc = str(row['DESCRIPTION']).lower()

    # 1. Min Temperature 
    if pd.notna(min_t) and min_t > 0:
        row['Operating Temperature-Min (Cel)'] = -abs(min_t)

    # 2. Correct max andf min
    if min_t == 55.0 or (pd.notna(min_t) and pd.notna(max_t) and min_t == max_t):
        if '125' in desc:
            row['Operating Temperature-Max (Cel)'] = 125.0
            row['Operating Temperature-Min (Cel)'] = -55.0
        elif '85' in desc:
            row['Operating Temperature-Max (Cel)'] = 85.0
            row['Operating Temperature-Min (Cel)'] = -40.0 or -55.0 # Logic based on data
        elif '105' in desc:
            row['Operating Temperature-Max (Cel)'] = 105.0

    # 3. Sanity check
    if pd.notna(row['Operating Temperature-Min (Cel)']) and pd.notna(row['Operating Temperature-Max (Cel)']):
        if row['Operating Temperature-Min (Cel)'] > row['Operating Temperature-Max (Cel)']:
            # Swap values
            row['Operating Temperature-Min (Cel)'], row['Operating Temperature-Max (Cel)'] = \
            row['Operating Temperature-Max (Cel)'], row['Operating Temperature-Min (Cel)']

    return row

# Fix apply
df_structured = df_structured.apply(fix_temperature_logic, axis=1)

#  Audit check 
print("--- ✅ Temperature Fixed! ---")
print(df_structured[['Operating Temperature-Min (Cel)', 'Operating Temperature-Max (Cel)']].describe())
print("\nUnique Min Values:", df_structured['Operating Temperature-Min (Cel)'].unique())

--- ✅ Temperature Fixed! ---
       Operating Temperature-Min (Cel)  Operating Temperature-Max (Cel)
count                       636.000000                       707.000000
mean                        -54.363208                        76.117397
std                           3.026679                        31.542436
min                         -55.000000                        55.000000
25%                         -55.000000                        55.000000
50%                         -55.000000                        55.000000
75%                         -55.000000                       125.000000
max                         -40.000000                       125.000000

Unique Min Values: [-55. -40.  nan]


In [ ]:
# percent data missing
null_percentage = (df_structured.isnull().sum() / len(df_structured)) * 100

# sorted by percentage
print("--- 📊 Null Values Percentage Report ---")
report = null_percentage[null_percentage > 0].sort_values(ascending=False)

if report.empty:
    print("Mubarak ho! Kisi bhi column mein null values nahi hain.")
else:
    print(report.map(lambda n: f"{n:.2f}%"))

# Visual confirmation 
print(f"\nTotal Rows in Dataset: {len(df_structured)}")

--- 📊 Null Values Percentage Report ---
Rated Voltage(AC) (V)              93.59%
Rated Breaking Capacity (A)        89.98%
Pre-arcing time-Min (ms)           89.18%
Maximum Power Dissipation          74.65%
Additional Feature                 67.54%
Maximum DC Voltage Rating          55.11%
Rated Voltage(DC) (V)              53.71%
Code                               44.39%
Operating Temperature-Min (Cel)    36.27%
Joule-integral-Nom (J)             33.57%
Characteristic                     32.67%
Mounting Feature                   31.96%
Operating Temperature-Max (Cel)    29.16%
Maximum AC Voltage Rating          26.45%
Product Diameter                   24.45%
Number of Terminals                24.05%
Attribut1                          22.85%
Material                           22.75%
Mounting                           22.44%
LC Risk                            21.24%
Application                        17.74%
Product Length                     14.43%
Size                               1

In [23]:
import re

def final_mining(row):
    desc = str(row['DESCRIPTION']).upper() if pd.notna(row['DESCRIPTION']) else ""
    
    # 1. Recover Rated Current (e.g., "1.6A", "10 AMP")
    if pd.isna(row['Rated Current (A)']):
        match = re.search(r'(\d+\.?\d*)\s*(A|AMP)\b', desc)
        if match: row['Rated Current (A)'] = float(match.group(1))

    # 2. Recover Rated Voltage & AC/DC Distinction
    if pd.isna(row['Rated Voltage (V)']):
        v_match = re.search(r'(\d+)\s*(V|VAC|VDC)\b', desc)
        if v_match: row['Rated Voltage (V)'] = float(v_match.group(1))
    
    if pd.isna(row['Rated Voltage(AC) (V)']) and 'VAC' in desc:
        vac_match = re.search(r'(\d+)\s*VAC', desc)
        if vac_match: row['Rated Voltage(AC) (V)'] = float(vac_match.group(1))

    # 3. Recover Breaking Capacity (e.g., "1500A (IR)", "35A @ 250V")
    if pd.isna(row['Rated Breaking Capacity (A)']):
        bc_match = re.search(r'(\d+)\s*A\s*\(IR\)', desc)
        if bc_match: row['Rated Breaking Capacity (A)'] = float(bc_match.group(1))

    # 4. Recover Dimensions (e.g., "5X20MM")
    if pd.isna(row['Product Diameter']) or pd.isna(row['Product Length']):
        dim_match = re.search(r'(\d+\.?\d*)\s*[X*]\s*(\d+\.?\d*)\s*MM', desc)
        if dim_match:
            if pd.isna(row['Product Diameter']): row['Product Diameter'] = float(dim_match.group(1))
            if pd.isna(row['Product Length']): row['Product Length'] = float(dim_match.group(2))

    # 5. Recover Material (Ceramic/Glass)
    if pd.isna(row['Material']):
        if 'CERAMIC' in desc: row['Material'] = 'CERAMIC'
        elif 'GLASS' in desc: row['Material'] = 'GLASS'

    return row

# Apply the mining
df_final_clean = df_structured.apply(final_mining, axis=1)

# Final Status Check
print("--- 🚀 Recovery Complete! ---")
new_nulls = df_final_clean[['Rated Current (A)', 'Rated Voltage (V)', 'Product Diameter', 'Material', 'Rated Breaking Capacity (A)']].isnull().sum()
print("New Null Counts:\n", new_nulls)

--- 🚀 Recovery Complete! ---
New Null Counts:
 Rated Current (A)               53
Rated Voltage (V)               80
Product Diameter               244
Material                       223
Rated Breaking Capacity (A)    341
dtype: int64


In [ ]:
# 1. Fill Categorical Nulls with 'UNKNOWN'
categorical_to_fill = ['Material', 'Characteristic', 'Mounting', 'Application_Primary', 'Attribut1', 'LC Risk']
for col in categorical_to_fill:
    if col in df_final_clean.columns:
        df_final_clean[col] = df_final_clean[col].fillna('UNKNOWN')

# 2. Fill Numeric Nulls with Median
numeric_to_fill = ['Rated Current (A)', 'Rated Voltage (V)', 'Product Diameter', 'Product Length', 'Rated Breaking Capacity (A)']
for col in numeric_to_fill:
    if col in df_final_clean.columns:
        median_val = df_final_clean[col].median()
        df_final_clean[col] = df_final_clean[col].fillna(median_val)

# 3. Final Drop
cols_to_final_drop = ['Additional Feature', 'Code', 'Joule-integral-Nom (J)'] # Joule agar analysis ka part nahi hai
df_final_clean = df_final_clean.drop(columns=[c for c in cols_to_final_drop if c in df_final_clean.columns])

print("✅ Final Imputation Complete. Zero Nulls remaining!")
print(df_final_clean.isnull().sum().sum())

✅ Final Imputation Complete. Zero Nulls remaining!
4925


In [ ]:
# 1. Statistical Plausibility 
print("--- 📊 Level 1: Statistical Check ---")
# Rated Current 
logic_check = df_final_clean[
    (df_final_clean['Rated Current (A)'] <= 0) | 
    (df_final_clean['Rated Voltage (V)'] < 12) |
    (df_final_clean['Product Diameter'] <= 0)
]

if logic_check.empty:
    print("✅ Logic Pass: No zero/negative values in critical electrical specs.")
else:
    print(f"⚠️ Warning: Found {len(logic_check)} rows with suspicious 0 or negative values.")

# 2. Imputation Impact 
print("\n--- ⚖️ Level 2: Imputation Impact ---")

print(df_final_clean[['Rated Current (A)', 'Rated Voltage (V)', 'Product Diameter']].describe().loc[['mean', '50%']])

# 3. Categorical Consistency
print("\n--- 🏷️ Level 3: Categorical Consistency ---")
# Check if 'UNKNOWN' is dominating any column
for col in ['Material', 'Characteristic']:
    unknown_count = (df_final_clean[col] == 'UNKNOWN').sum()
    print(f"{col} 'UNKNOWN' entries: {unknown_count} ({ (unknown_count/len(df_final_clean)*100):.1f}%)")

--- 📊 Level 1: Statistical Check ---
✅ Logic Pass: No zero/negative values in critical electrical specs.

--- ⚖️ Level 2: Imputation Impact ---
      Rated Current (A)  Rated Voltage (V)  Product Diameter
mean           3.544954         155.637275          6.166503
50%            2.000000         125.000000          6.100000

--- 🏷️ Level 3: Categorical Consistency ---
Material 'UNKNOWN' entries: 223 (22.3%)
Characteristic 'UNKNOWN' entries: 326 (32.7%)


In [ ]:
# df_final_clean current missing percentage report
null_report = (df_final_clean.isnull().sum() / len(df_final_clean)) * 100

missing_status = null_report[null_report > 0].sort_values(ascending=False)

print("--- 🔍 df_final_clean: Current Missing Percentage ---")
if missing_status.empty:
    print("✅ Great! Is dataframe mein ab ek bhi missing value nahi hai.")
else:
    print(missing_status.map(lambda x: f"{x:.2f}%"))
    
print(f"\nTotal rows: {len(df_final_clean)}")

--- 🔍 df_final_clean: Current Missing Percentage ---
Pre-arcing time-Min (ms)           89.18%
Maximum Power Dissipation          74.65%
Maximum DC Voltage Rating          55.11%
Rated Voltage(DC) (V)              53.71%
Rated Voltage(AC) (V)              43.39%
Operating Temperature-Min (Cel)    36.27%
Mounting Feature                   31.96%
Operating Temperature-Max (Cel)    29.16%
Maximum AC Voltage Rating          26.45%
Number of Terminals                24.05%
Application                        17.74%
Size                               11.82%
dtype: object

Total rows: 998


In [ ]:
# 1. Dropping low-quality / high-null columns
cols_to_drop = [
    'Pre-arcing time-Min (ms)', 
    'Maximum Power Dissipation', 
    'Number of Terminals',
    'Additional Feature' 
]
df_final_clean = df_final_clean.drop(columns=[c for c in cols_to_drop if c in df_final_clean.columns])

# 2. Smart Imputation 
for col in ['Operating Temperature-Min (Cel)', 'Operating Temperature-Max (Cel)']:
    if col in df_final_clean.columns:
        df_final_clean[col] = df_final_clean[col].fillna(df_final_clean[col].median())

# Categorical: 'NOT SPECIFIED' label 
for col in ['Mounting Feature', 'Application', 'Size', 'Material', 'Characteristic']:
    if col in df_final_clean.columns:
        df_final_clean[col] = df_final_clean[col].fillna('NOT SPECIFIED')

# 3. Save the Cleanest Version
df_final_clean.to_csv("data_clean.csv", index=False, encoding='utf-8-sig')

print("✅ Final Clean Completed!")
print(f"Columns Remaining: {list(df_final_clean.columns)}")
print(f"Remaining Nulls: {df_final_clean.isnull().sum().sum()}")

✅ Final Clean Completed!
Columns Remaining: ['ID', 'DESCRIPTION', 'Attribut1', 'Application', 'Characteristic', 'Material', 'Size', 'LC Risk', 'Maximum AC Voltage Rating', 'Maximum DC Voltage Rating', 'Mounting', 'Mounting Feature', 'Operating Temperature-Max (Cel)', 'Operating Temperature-Min (Cel)', 'Product Diameter', 'Product Length', 'Rated Breaking Capacity (A)', 'Rated Current (A)', 'Rated Voltage (V)', 'Rated Voltage(AC) (V)', 'Rated Voltage(DC) (V)']
Remaining Nulls: 1783


In [ ]:
# Detailed Null Report
null_counts = df_final_clean.isnull().sum()
null_report = pd.DataFrame({
    'Missing Values': null_counts,
    'Percentage': (null_counts / len(df_final_clean) * 100).round(2)
})

print("--- 🔍 Missing Data Inventory ---")
print(null_report[null_report['Missing Values'] > 0].sort_values(by='Missing Values', ascending=False))


--- 🔍 Missing Data Inventory ---
                           Missing Values  Percentage
Maximum DC Voltage Rating             550       55.11
Rated Voltage(DC) (V)                 536       53.71
Rated Voltage(AC) (V)                 433       43.39
Maximum AC Voltage Rating             264       26.45
